In [ ]:
# -*- coding: utf-8 -*-  
"""
Closed-Form NCA with RK4 Self-Organizing Inference
Live inference during training (tested in JupyterLab)
"""  

import torch, torch.nn as nn, torch.nn.functional as F
import time, asyncio, cv2, io, numpy as np, traceback
import ipywidgets as widgets
from IPython.display import display
from collections import deque
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt

try: import clip
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "-q", "git+https://github.com/openai/CLIP.git"])
    import clip

torch.backends.cudnn.benchmark = True  
device = 'cuda' if torch.cuda.is_available() else 'cpu'  


clip_model, _ = clip.load('RN101', device=device, jit=False)
clip_model.eval().float()
for p in clip_model.parameters(): p.requires_grad_(False)

CLIP_MEAN = torch.tensor([0.4814, 0.4578, 0.4082], device=device).view(1,3,1,1)
CLIP_STD  = torch.tensor([0.2686, 0.2613, 0.2758], device=device).view(1,3,1,1)
def clip_norm(x): return (x - CLIP_MEAN) / CLIP_STD

def random_crops(img, n=2, size=224):
    B, C, H, W = img.shape
    scales = torch.empty(n, device=img.device).uniform_(0.7, 1.2) 
    tx = torch.empty(n, device=img.device).uniform_(-0.1, 0.1)
    ty = torch.empty(n, device=img.device).uniform_(-0.1, 0.1)
    flip = (torch.rand(n, device=img.device) < 0.5).float() * 2 - 1
    
    theta = torch.zeros(n, 2, 3, device=img.device)
    theta[:,0,0] = scales * flip; theta[:,1,1] = scales
    theta[:,0,2] = tx; theta[:,1,2] = ty
    
    grid = F.affine_grid(theta, (n, C, size, size), align_corners=False)
    expanded = img.unsqueeze(1).expand(-1, n, -1, -1, -1).reshape(B * n, C, H, W)
    return F.grid_sample(expanded, grid.repeat(B, 1, 1, 1), padding_mode='reflection', align_corners=False)

def get_structured_noise(B, C, size, device):
    grid_size = 8
    z = torch.randn(B, C, grid_size, grid_size, device=device)
    z_pad = F.pad(z, (2, 2, 2, 2), mode='circular')
    scale = size // grid_size
    z_up = F.interpolate(z_pad, scale_factor=scale, mode='bicubic', align_corners=False)
    pad_scale = 2 * scale
    return z_up[:, :, pad_scale:-pad_scale, pad_scale:-pad_scale] * 0.5

def get_circular_noise(B, C, size, device):
    noise = torch.randn(B, C, size, size, device=device)
    fy = torch.fft.fftfreq(size, device=device).view(-1, 1)
    fx = torch.fft.fftfreq(size, device=device).view(1, -1)
    # Smooth the noise in frequency space so edges match perfectly
    mask = torch.exp(-100.0 * (fx**2 + fy**2))
    smooth = torch.fft.ifft2(torch.fft.fft2(noise) * mask).real
    smooth = (smooth - smooth.mean(dim=(2,3), keepdim=True)) / (smooth.std(dim=(2,3), keepdim=True) + 1e-5)
    return smooth * 0.5

get_noise = get_circular_noise


_act = {}
def get_hook(name):
    def _hook(mod, inp, out): _act[name] = out
    return _hook

for layer_name in ['layer1', 'layer2', 'layer3', 'layer4']:
    dict(clip_model.visual.named_modules())[layer_name].register_forward_hook(get_hook(layer_name))


class ClosedFormNCA(nn.Module):  
    def __init__(self, in_ch=12, latent_ch=48, kernel_size=5):  
        super().__init__()  
        self.C = latent_ch  
        self.k_size = kernel_size
        self.shift = kernel_size // 2 

        self.lift = nn.Sequential(nn.Conv2d(in_ch, latent_ch, 1), nn.GELU(),
                                  nn.Conv2d(latent_ch, latent_ch, 1), nn.GELU(),
                                  nn.Conv2d(latent_ch, latent_ch, 1))
                                  
        self.project = nn.Sequential(nn.Conv2d(latent_ch, latent_ch, 1), nn.GELU(),                                  
                                     nn.Conv2d(latent_ch, latent_ch, 1), nn.GELU(),
                                     nn.Conv2d(latent_ch, 3, 1), nn.Sigmoid())
        
        self.U_raw = nn.Parameter(torch.randn(latent_ch, latent_ch) * 0.1)
        self.spatial_kernel = nn.Parameter(torch.randn(latent_ch, 1, kernel_size, kernel_size) * 0.05)
        self.damping_raw = nn.Parameter(torch.zeros(latent_ch) - 2.0) 

    def forward(self, t, x0):  
        z0 = self.lift(x0)
        orig_dtype = z0.dtype
        
        Z0 = torch.fft.fft2(z0.to(torch.float32))
        
        U = torch.matrix_exp(self.U_raw.to(torch.float32) - self.U_raw.T.to(torch.float32))
        U_complex = U.to(torch.complex64) 
        
        k = self.spatial_kernel.to(torch.float32)
        pad_w, pad_h = x0.shape[3] - self.k_size, x0.shape[2] - self.k_size
        k_pad = torch.roll(F.pad(k, (0, pad_w, 0, pad_h)), (-self.shift, -self.shift), (2, 3))
        g_hat = torch.fft.fft2(k_pad).squeeze(1)

        decay = F.softplus(self.damping_raw.to(torch.float32)).view(1, -1, 1, 1) * 0.0
        g_stable = (g_hat.real - g_hat.real.amax(dim=(1, 2), keepdim=True) - decay) + 1j * g_hat.imag

        Z_rot = torch.einsum('cd, bdhw -> bchw', U_complex.T, Z0)
        Z_t = Z_rot * torch.exp(t.to(torch.float32) * g_stable)
        Z_final = torch.einsum('cd, bdhw -> bchw', U_complex, Z_t)

        xt = torch.fft.ifft2(Z_final).real.to(orig_dtype)
        return xt, self.project(xt)

    def step_latent_rk4(self, z, dt=0.05):
        orig_dtype = z.dtype
        z_f32 = z.to(torch.float32)
        
        k = self.spatial_kernel.to(torch.float32)
        pad_w, pad_h = z.shape[3] - self.k_size, z.shape[2] - self.k_size
        k_pad = torch.roll(F.pad(k, (0, pad_w, 0, pad_h)), (-self.shift, -self.shift), (2, 3))
        
        max_real = torch.fft.fft2(k_pad).squeeze(1).real.amax(dim=(1, 2)).view(self.C)
        decay = F.softplus(self.damping_raw.to(torch.float32)).view(self.C) * 0.0

        k_stable = k.clone()
        k_stable[:, 0, self.shift, self.shift] -= (max_real + decay)

        k_conv = torch.flip(k_stable, [2, 3])
        U = torch.matrix_exp(self.U_raw.to(torch.float32) - self.U_raw.T.to(torch.float32))

        def dynamics(x):
            x_rot = F.pad(torch.einsum('cd, bdhw -> bchw', U.T, x), 
                          (self.shift, self.shift, self.shift, self.shift), mode='circular')
            dx_rot = F.conv2d(x_rot, k_conv, groups=self.C)
            return torch.einsum('cd, bdhw -> bchw', U, dx_rot)

        k1 = dynamics(z_f32)
        k2 = dynamics(z_f32 + 0.5 * dt * k1)
        k3 = dynamics(z_f32 + 0.5 * dt * k2)
        k4 = dynamics(z_f32 + dt * k3)

        return (z_f32 + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)).to(orig_dtype)


loss_log, best_loss = [], float('inf')

layer_dropdown = widgets.Dropdown(options=['layer1', 'layer2', 'layer3', 'layer4'], value='layer2', description='Layer:', layout=widgets.Layout(width='180px'))
channel_slider = widgets.IntSlider(min=0, max=511, value=42, description='Channel:', layout=widgets.Layout(width='500px'))
dd_toggle = widgets.Checkbox(value=False, description='DeepDream', layout=widgets.Layout(width='300px'))
btn_restart_train = widgets.Button(description='💣 Hard Reset Model', button_style='danger', icon='bomb', layout=widgets.Layout(width='180px', margin='0 0 0 15px'))

def update_channel_bounds(*args):
    ch_map = {'layer1': 256, 'layer2': 512, 'layer3': 1024, 'layer4': 2048}
    channel_slider.max = ch_map[layer_dropdown.value] - 1
layer_dropdown.observe(update_channel_bounds, 'value')

def update_dd_mode(*args):
    channel_slider.disabled = dd_toggle.value
dd_toggle.observe(update_dd_mode, 'value')

def reset_graph(*args):
    global loss_log, best_loss
    loss_log, best_loss = [], float('inf')
    
layer_dropdown.observe(reset_graph, 'value')
channel_slider.observe(reset_graph, 'value')
dd_toggle.observe(reset_graph, 'value')

train_controls_ui = widgets.HBox([layer_dropdown, channel_slider, dd_toggle, btn_restart_train], layout=widgets.Layout(margin='10px 0 0 0', align_items='center'))

class AsyncDOR:  
    def __init__(self, shader_func, reset_cb=None, res=128):  
        self.shader_func, self.reset_cb = shader_func, reset_cb
        self._running = self._paused = False  
        self.t = self._pause_offset = 0.0  
        
        self.img_widget = widgets.Image(value=cv2.imencode('.jpg', np.zeros((res, res*2, 3), np.uint8))[1].tobytes(), format='jpeg', width=512, height=256)  
        self.fps_label = widgets.Label(value='Ready')  
        self.legend_label = widgets.HTML(value=f"<div style='display:flex; width:512px; text-align:center; font-family:sans-serif; font-size:14px; font-weight:bold; color:#ccc; background:#222; padding:6px 0; border-radius:4px 4px 0 0;'><div style='flex:1;'>⚡ Closed-Form O(1)</div><div style='flex:1;'>🦠 RK4 Cellular Automaton</div></div>")  

        self.btn_toggle = widgets.ToggleButton(icon='play', button_style='success', layout=widgets.Layout(width='32px'))  
        self.btn_toggle.observe(self._on_toggle, names='value')  
        self.btn_pause = widgets.ToggleButton(icon='pause', button_style='warning', layout=widgets.Layout(width='32px'))  
        self.btn_pause.observe(self._on_pause, names='value')  
        self.btn_reset = widgets.Button(icon='refresh', button_style='info', layout=widgets.Layout(width='32px'))  
        self.btn_reset.on_click(lambda _: self.reset_time())

        self.ui = widgets.VBox([self.legend_label, self.img_widget, widgets.HBox([self.btn_toggle, self.btn_pause, self.btn_reset, self.fps_label], layout=widgets.Layout(align_items='center'))])  

    def _task_error_callback(self, task):
        try: task.result()
        except asyncio.CancelledError: pass
        except Exception as e:
            print("\nFATAL ERROR:\n", traceback.format_exc())
            try: stats_label.value = f"CRASH: {type(e).__name__} (See console)"
            except: pass

    def _on_toggle(self, change):  
        if change['new']:  
            self._running = True; self.btn_toggle.icon = 'stop'; self.btn_toggle.button_style = 'danger'
            self._unpause_wall = time.time()
            self._task = asyncio.create_task(self._loop()); self._task.add_done_callback(self._task_error_callback)
            self.train_task = asyncio.create_task(train_step()); self.train_task.add_done_callback(self._task_error_callback)
        else:  
            self._running = False; self.btn_toggle.icon = 'play'; self.btn_toggle.button_style = 'success'
            if hasattr(self, '_task'): self._task.cancel()
            if hasattr(self, 'train_task'): self.train_task.cancel()

    def _on_pause(self, change):  
        self._paused = change['new']
        if self._paused: self._pause_offset = self.t; self.btn_pause.icon = 'play'
        else: self._unpause_wall = time.time(); self.btn_pause.icon = 'pause'

    def reset_time(self):  
        self.t = self._pause_offset = 0.0; self._unpause_wall = time.time()  
        if self.reset_cb: self.reset_cb()

    async def _loop(self):  
        frames, last_send = deque(maxlen=30), 0.0  
        while self._running:  
            if self._paused: await asyncio.sleep(0.05); continue  
            t0 = time.time()
            
            new_t = self._pause_offset + (t0 - self._unpause_wall) * 2.0 
            dt = new_t - self.t
            self.t = new_t
            
            with torch.no_grad(): img_np = (self.shader_func(self.t, dt).clamp(0, 1).mul_(255)).byte().cpu().numpy()  
            if t0 - last_send >= 0.03:  
                self.img_widget.value = cv2.imencode('.jpg', img_np[:, :, ::-1], [int(cv2.IMWRITE_JPEG_QUALITY), 60])[1].tobytes()  
                last_send = t0  
            
            frames.append(time.time() - t0)  
            if len(frames) == 30: self.fps_label.value = f"FPS: {1.0/(sum(frames)/30):.0f} | t={self.t:.2f}"  
            await asyncio.sleep(0.01)  

# Model init
ca_model = ClosedFormNCA(in_ch=12, latent_ch=48, kernel_size=5).to(device)  
optimizer = torch.optim.AdamW(ca_model.parameters(), lr=4e-3, weight_decay=1e-4) 
scaler = torch.amp.GradScaler('cuda')  

fig, ax = plt.subplots(figsize=(6, 2.5))  
ax.text(0.5, 0.5, "Waiting for loss data...", ha='center', va='center', color='gray')  
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)  
fig.tight_layout(pad=1.0)  
buf = io.BytesIO(); fig.savefig(buf, format='png', bbox_inches='tight'); plt.close(fig)  

graph_widget = widgets.Image(value=buf.getvalue(), format='png', width=500, height=200) 
stats_label = widgets.Label(value="Waiting...", layout=widgets.Layout(margin='0 0 0 15px', font_weight='bold', color='#FFD700'))

display_x0 = get_noise(1, 12, 128, device)
z_rk4 = ca_model.lift(display_x0)

def reset_latent():
    global display_x0, z_rk4
    display_x0 = get_noise(1, 12, 128, device)
    z_rk4 = ca_model.lift(display_x0)

def hard_reset_model(*args):
    """Destroys current physics and re-initializes model/optimizers from scratch."""
    global ca_model, optimizer, scaler, loss_log, best_loss
    
    ca_model = ClosedFormNCA(in_ch=12, latent_ch=48, kernel_size=5).to(device)  
    optimizer = torch.optim.AdamW(ca_model.parameters(), lr=4e-3, weight_decay=1e-4) 
    scaler = torch.amp.GradScaler('cuda')  
    
    loss_log, best_loss = [], float('inf')
    reset_latent()
    
    fig, ax = plt.subplots(figsize=(6, 2.5))  
    ax.text(0.5, 0.5, "Universe Destroyed. Rebuilding...", ha='center', va='center', color='red')  
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)  
    fig.tight_layout(pad=1.0)  
    buf = io.BytesIO(); fig.savefig(buf, format='png', bbox_inches='tight'); plt.close(fig)  
    graph_widget.value = buf.getvalue()
    stats_label.value = "⚡ Physics Re-rolled | 🏆 Loss: inf"

btn_restart_train.on_click(hard_reset_model)

def shader(t, dt):  
    global z_rk4
    t_tensor = torch.tensor([t], device=device).view(1, 1, 1, 1)
    _, rgb_cf = ca_model(t_tensor, display_x0)  
    
    sub_steps = max(1, int(dt / 0.1))
    for _ in range(sub_steps): z_rk4 = ca_model.step_latent_rk4(z_rk4, dt=dt/sub_steps)
    
    rgb_cf, rgb_rk4 = rgb_cf[0].permute(1, 2, 0), ca_model.project(z_rk4)[0].permute(1, 2, 0)
    return torch.cat([rgb_cf, rgb_rk4], dim=1)  

async def train_step():  
    global loss_log, best_loss
    try:
        t0 = time.time()
        while dor._running: 
            t_train = torch.empty(2, 1, 1, 1, device=device).uniform_(4.0, 16.0)
            x0 = get_noise(2, 12, 128, device)
            
            with torch.amp.autocast('cuda'):  
                _, rgb = ca_model(t_train, x0)
                
                views = random_crops(rgb, n=2) 
                views = views + torch.randn_like(views) * 0.02 
                
                _ = clip_model.encode_image(clip_norm(views)) 
                
                current_layer = layer_dropdown.value
                current_acts = _act[current_layer]
                
                if dd_toggle.value:
                    fv_loss = -current_acts.square().mean() * 5.0
                    title_str = f"Targeting {current_layer} (DeepDream)"
                else:
                    target_ch = channel_slider.value
                    fv_loss = -current_acts[:, target_ch].mean() * 5.0
                    title_str = f"Targeting {current_layer} C{target_ch}"
                
                l2_reg = rgb.square().mean() * 0.1
                tv_loss = ((rgb[:,:,:,:-1] - rgb[:,:,:,1:]).square().mean() + (rgb[:,:,:-1,:] - rgb[:,:,1:,:]).square().mean()) * 0.005
                loss = fv_loss + l2_reg + tv_loss

            optimizer.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(optimizer)  
            torch.nn.utils.clip_grad_norm_(ca_model.parameters(), 1.0)  
            scaler.step(optimizer); scaler.update()
            
            current_loss = fv_loss.item()
            loss_log.append(current_loss)
            if current_loss < best_loss: best_loss = current_loss

            stats_label.value = f"⚡ {1.0 / max(time.time() - t0, 1e-4):.0f} it/s | 🏆 Loss: {best_loss:.2f}"  
            t0 = time.time()
            
            if len(loss_log) > 0 and len(loss_log) % 15 == 0:  
                fig, ax = plt.subplots(figsize=(6, 2.5))  
                ax.plot(loss_log[-500:], color='#1f77b4', linewidth=1.5); ax.set_title(title_str, fontsize=10)  
                ax.grid(True, linestyle='--', alpha=0.5); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)  
                fig.tight_layout(pad=1.0)  
                buf = io.BytesIO(); fig.savefig(buf, format='png', bbox_inches='tight'); plt.close(fig)  
                graph_widget.value = buf.getvalue() 

            await asyncio.sleep(0.001)  
    except asyncio.CancelledError: pass 

dor = AsyncDOR(shader_func=shader, reset_cb=reset_latent)  
layout_block = widgets.VBox([
    dor.ui, 
    train_controls_ui, 
    widgets.HBox([graph_widget, stats_label], layout=widgets.Layout(align_items='center'))
])
display(layout_block)  
dor.btn_toggle.value = True